# Exercises XP Gold ? Prompt Engineering
Last Updated: October 7th, 2025
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Craft precise prompts for complex tasks.
- Debug and refine chain-of-thought (CoT) reasoning.
- Select prompt patterns for real-world applications and reduce hallucinations.
- Design chained prompts with conditional logic, bias mitigation, and simulated memory.

## What you'll build
- Corrected CoT prompts and answers.
- Domain-specific prompt patterns and multi-step pipelines.
- Fair role-based prompts and conversational agents with memory.

## Helper: Run a prompt
Uses `ollama run` if available (set `OLLAMA_MODEL` to override, default: `llama3`). If Ollama is missing or the call fails, the helper prints the prompt in dry-run mode instead of crashing.


In [ ]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 1: Debug a Faulty Chain-of-Thought
Goal: Fix the CoT and final answer. Original prompt had math/logic issues for the pencil change problem.

### Tasks
- Find the reasoning mistake.
- Rewrite the CoT prompt with correct steps.
- Provide the correct answer.

In [1]:
# TODO: describe the mistake
cot_issue = (
    "The arithmetic is incorrect because 6 pencils at 0.75 dollars each cost 4.50 dollars, not 4.75 dollars. "
    "The error occurs during the multiplication step, which leads to an incorrect change amount."
)
cot_issue


'The arithmetic is incorrect because 6 pencils at 0.75 dollars each cost 4.50 dollars, not 4.75 dollars. The error occurs during the multiplication step, which leads to an incorrect change amount.'

In [2]:
# TODO: rewrite the CoT prompt
fixed_cot_prompt = (
    "## Task "
    "Solve the following problem step by step with correct arithmetic. "
    "## Problem "
    "A shop sells pencils at 0.75 dollars each. Alice buys 6 pencils and pays with a 5 dollar bill. "
    "## Reasoning Steps "
    "First calculate the total cost of the pencils. "
    "Then subtract the total cost from the amount paid. "
    "Ensure each calculation is correct before moving to the next step. "
    "## Output "
    "Provide the final amount of change."
)
fixed_cot_prompt


'## Task Solve the following problem step by step with correct arithmetic. ## Problem A shop sells pencils at 0.75 dollars each. Alice buys 6 pencils and pays with a 5 dollar bill. ## Reasoning Steps First calculate the total cost of the pencils. Then subtract the total cost from the amount paid. Ensure each calculation is correct before moving to the next step. ## Output Provide the final amount of change.'

In [3]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


In [4]:
# TODO: final correct answer
correct_change = "Alice receives 0.50 dollars in change."
correct_change


'Alice receives 0.50 dollars in change.'

## Exercise 2: Choose the Right Prompt Pattern
Goal: Pick and justify a prompt pattern for support ticket classification (Billing Issue, Technical Support, Account Access, Other).

### Tasks
- Choose a pattern (Zero-Shot, Few-Shot, IAP, LoT, etc.).
- Write a complete prompt using that pattern.
- Justify the choice (ambiguity, consistency, generalization).

In [5]:
# TODO: choose pattern and prompt
chosen_pattern = "Few-Shot"

classification_prompt = (
    "## Role "
    "You are a customer support classification assistant. "
    "## Task "
    "Categorize the customer message into one of the following labels Billing Issue Technical Support Account Access or Other. "
    "## Examples "
    "Message I was charged twice for my subscription Label Billing Issue. "
    "Message I cannot log into my account Label Account Access. "
    "Message The app crashes when I open it Label Technical Support. "
    "## Instruction "
    "Now classify the following customer message using the same labels. "
    "Return only the label."
)

justification = (
    "Few shot prompting is best because it improves consistency in ambiguous cases, "
    "helps the model generalize from real examples, "
    "and reduces misclassification compared to zero shot approaches."
)

classification_prompt


'## Role You are a customer support classification assistant. ## Task Categorize the customer message into one of the following labels Billing Issue Technical Support Account Access or Other. ## Examples Message I was charged twice for my subscription Label Billing Issue. Message I cannot log into my account Label Account Access. Message The app crashes when I open it Label Technical Support. ## Instruction Now classify the following customer message using the same labels. Return only the label.'

In [6]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 3: Use AlignedCoT to Compare Reasoning Paths
Goal: Two reasoning paths + comparison for the flower pot cost.

### Tasks
- Build AlignedCoT with at least two distinct reasoning structures.
- Add a comparison step to select the consistent answer.

In [7]:
# TODO: aligned CoT prompt
aligned_cot_prompt = (
    "## Task "
    "Calculate the total cost of the flower pots using two different reasoning paths. "
    "## Reasoning Path One "
    "Calculate the cost by grouping each pot type separately and summing the totals. "
    "## Reasoning Path Two "
    "Calculate the cost by summing all quantities first and then multiplying by their respective prices. "
    "## Comparison "
    "Compare the results from both reasoning paths and select the consistent final answer. "
    "## Output "
    "Provide the final total cost."
)
aligned_cot_prompt


'## Task Calculate the total cost of the flower pots using two different reasoning paths. ## Reasoning Path One Calculate the cost by grouping each pot type separately and summing the totals. ## Reasoning Path Two Calculate the cost by summing all quantities first and then multiplying by their respective prices. ## Comparison Compare the results from both reasoning paths and select the consistent final answer. ## Output Provide the final total cost.'

In [8]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 4: Design a Multi-Step Document Pipeline
Goal: Three-stage pipeline (domain detect -> contributions -> follow-up question) with conditional/context chaining.

### Tasks
- Draft prompt template for each stage.
- Note where to chain context or branch conditionally.

In [9]:
# TODO: stage prompts
stage1_domain = (
    "## Task "
    "Identify the academic domain of the following research paper abstract. "
    "## Output "
    "Return one domain such as biology physics or computer science."
)

stage2_contrib = (
    "## Task "
    "Extract the main research contributions from the abstract provided. "
    "## Instructions "
    "Summarize the contributions in clear bullet points."
)

stage3_followup = (
    "## Task "
    "Generate one meaningful follow up research question based on the extracted contributions. "
    "## Instruction "
    "Ensure the question aligns with the identified domain."
)

chaining_notes = (
    "The domain identified in stage one should be passed to stages two and three. "
    "If the domain is interdisciplinary branch to multiple contribution summaries. "
    "The extracted contributions should be injected as context when generating the follow up question."
)

stage1_domain, stage2_contrib, stage3_followup


('## Task Identify the academic domain of the following research paper abstract. ## Output Return one domain such as biology physics or computer science.',
 '## Task Extract the main research contributions from the abstract provided. ## Instructions Summarize the contributions in clear bullet points.',
 '## Task Generate one meaningful follow up research question based on the extracted contributions. ## Instruction Ensure the question aligns with the identified domain.')

## Exercise 5: Role Prompting to Reduce Bias
Goal: Fair career recommendations.

### Tasks
- Write a basic prompt (may be biased).
- Write a revised role-based prompt to reduce bias.
- Explain how the role prompt improves fairness.

In [10]:
# TODO: basic and revised prompts
biased_prompt = (
    "Suggest suitable career paths based on the user's skills and interests."
)

debiased_prompt = (
    "## Role "
    "You are a neutral and fair career guidance counselor. "
    "## Task "
    "Recommend career paths based solely on the user's skills interests and experience. "
    "## Fairness Rules "
    "Do not make assumptions based on gender age ethnicity or background. "
    "Ensure recommendations reflect equal opportunity and diverse career options."
)

fairness_explanation = (
    "The role based prompt improves fairness by explicitly instructing the model to avoid stereotypes, "
    "focus only on relevant qualifications, "
    "and apply the same evaluation criteria to all users."
)

debiased_prompt


"## Role You are a neutral and fair career guidance counselor. ## Task Recommend career paths based solely on the user's skills interests and experience. ## Fairness Rules Do not make assumptions based on gender age ethnicity or background. Ensure recommendations reflect equal opportunity and diverse career options."

In [11]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 6: Build a Conversational Agent with Context Memory
Goal: Add memory to a virtual health coach.

### Tasks
- Pick a memory technique (prior message passing, structured history, vector store retrieval).
- Show how you would structure past context.
- Write a prompt that injects that context for the next response.

In [12]:
# TODO: choose memory approach and build context + prompt
memory_approach = "Structured History"

past_context = (
    "## User Profile "
    "The user wants to improve sleep quality. "
    "## Previous Advice "
    "The coach previously recommended reducing screen time before bed and maintaining a consistent sleep schedule."
)

next_turn_prompt = (
    "## Role "
    "You are a virtual health coach. "
    "## Context "
    "Use the stored user profile and previous advice provided below. "
    "## Task "
    "Give personalized guidance to help the user further improve their sleep habits. "
    "## Past Context "
    "The user wants to improve sleep quality and has already been advised to reduce screen time and keep a consistent schedule."
)

next_turn_prompt


'## Role You are a virtual health coach. ## Context Use the stored user profile and previous advice provided below. ## Task Give personalized guidance to help the user further improve their sleep habits. ## Past Context The user wants to improve sleep quality and has already been advised to reduce screen time and keep a consistent schedule.'

In [13]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None
